# Inprocessing Bias Mitigation Techniques

## Load Data

In [ ]:
import sys
import os
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)


from fairlearn.metrics import (
    demographic_parity_ratio,
    equalized_odds_ratio,
    demographic_parity_difference,
    equalized_odds_difference
)

from fairlib.inprocessing.prejudice_remover import PrejudiceRemover
from fairlib.inprocessing.fauci import Fauci
from fairlib.dataframe import DataFrame

import torch.nn as nn

from utils.plot import plot_metrics, plot_metrics_grouped, get_mean_std, print_fairness_results_table

random_state = 42
np.random.seed(random_state)

In [ ]:
df_cleaned = pd.read_csv('cleaned_dataset.csv')

In [ ]:
features = [
    # Base Attributes
    "Sex_int",
    "Protected category",
    "Overall",
    "Technical Skills",
    "Standing/Position",
    "Comunication",
    "Maturity",
    "Dynamism",
    "Mobility",
    "English",
    "Italian Residence",
    "European Residence",
    "Age Range_int",
    "number_of_searches",
    # Custom Similarity Scores
    "experience_match_score",
    "current_salary_fit_score",
    "expected_salary_fit_score",
    "study_title_score",
    "professional_similarity_score",
    "study_area_score",
    "general_similarity_score",
    "Distance Residence - Akkodis HQ",
    "Distance Residence - Assumption HQ",
]

In [ ]:
protected_attributes = [
    'Sex_int', 'Protected category', 'Age Range_int',
    'Italian Residence', 'European Residence'
]

## Models

In [ ]:
base_model_lambda = lambda: nn.Sequential(
    nn.Linear(len(features), 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Sigmoid(),
)

models = {
    'prejudice_remover': lambda base_model, repair_level: PrejudiceRemover(
        torchModel=base_model,
        weight=repair_level
    ),
    'fauci': lambda base_model, repair_level: Fauci(
        torchModel=base_model,
        weight=repair_level  
    )
}

repair_levels = [0, 0.5, 1]
n_folds = 5

## Inprocessing

In [ ]:
def run_inprocessing(
    df, 
    n_folds, repair_levels,
    features, non_bool_cols, protected_attributes,
    models_dict
):
    df_selected_col = df[features + ['Hired']]
    dataset = pd.DataFrame(df_selected_col)

    bool_cols = dataset.select_dtypes(include='bool').columns
    non_bool_cols = dataset[features].columns.difference(bool_cols)

    dataset[bool_cols] = dataset[bool_cols].astype(int)

    results = defaultdict(list) 
    plot_data = defaultdict(lambda: defaultdict(dict))
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    for fold, (train_idx, test_idx) in tqdm(
        enumerate(kf.split(dataset, dataset['Hired'])), total=n_folds, desc='Folds...'
        ):

        train_df = dataset.iloc[train_idx].copy()
        test_df = dataset.iloc[test_idx].copy()

        imputer = SimpleImputer(strategy='mean')
        train_df = pd.DataFrame(imputer.fit_transform(train_df), columns=train_df.columns)
        test_df = pd.DataFrame(imputer.transform(test_df), columns=test_df.columns)
        
        train_df = DataFrame(train_df)
        test_df = DataFrame(test_df)
        
        scaler = StandardScaler()
        train_df[non_bool_cols] = scaler.fit_transform(train_df[non_bool_cols])
        test_df[non_bool_cols] = scaler.transform(test_df[non_bool_cols])
        
        for sensitive_attr in protected_attributes:
            train_df.sensitive = sensitive_attr
            train_df.targets = 'Hired'
            test_df.sensitive = sensitive_attr
            test_df.targets = 'Hired'
            y_train = train_df[['Hired']].values.ravel()

            for repair_level in repair_levels:
                for model_name, model_factory in models_dict.items():
                    ##################################################################################
                    base_model = base_model_lambda()
                    model = model_factory(base_model, repair_level)
                    model.fit(train_df[features], train_df[['Hired']], num_epochs=20, batch_size=32)

                    train_probs = model.predict(train_df[features]).numpy()
                    test_probs = model.predict(test_df[features]).numpy()
                    ########################################################################################
                    thresholds = np.linspace(0, 1, 101)
                    best_threshold = 0.5
                    best_f1 = 0.0
                    for thresh in thresholds:
                        train_preds = (train_probs > thresh).astype(int)
                        f1 = f1_score(y_train, train_preds, zero_division=0)
                        if f1 > best_f1:
                            best_f1 = f1
                            best_threshold = thresh
                
                    test_preds = (test_probs > best_threshold).astype(int)
                    y_test = test_df[['Hired']].values.ravel()
                    sens_test = test_df[sensitive_attr].values.ravel()

                    metrics = {
                        'accuracy': accuracy_score(y_test, test_preds),
                        'precision': precision_score(y_test, test_preds, zero_division=0),
                        'recall': recall_score(y_test, test_preds, zero_division=0),
                        'f1': f1_score(y_test, test_preds, zero_division=0),
                        'roc_auc': roc_auc_score(y_test, test_probs),
                        'demographic_parity_ratio': demographic_parity_ratio(
                            y_test, test_preds, sensitive_features=sens_test),
                        'equalized_odds_ratio': equalized_odds_ratio(
                            y_test, test_preds, sensitive_features=sens_test),
                        'demographic_parity_difference': demographic_parity_difference(
                            y_test, test_preds, sensitive_features=sens_test),
                        'equalized_odds_difference': equalized_odds_difference(
                            y_test, test_preds, sensitive_features=sens_test),
                    }

                    key = f"{sensitive_attr}_repair_{repair_level}_{model_name}"
                    results[key].append(metrics)
                    
    metrics_keys = list(metrics.keys())  
    for sensitive_attr in protected_attributes:
        for repair_level in repair_levels:
            for model_name in models_dict:
                key = f"{sensitive_attr}_repair_{repair_level}_{model_name}"
                fold_metrics = results.get(key, [])
                for metric in metrics_keys:
                    metric_list = [m[metric] for m in fold_metrics]
                    mean, std = get_mean_std(metric_list)
                    plot_data[model_name] [sensitive_attr][f"{metric}_mean_{repair_level}"] = mean
                    plot_data[model_name] [sensitive_attr][f"{metric}_std_{repair_level}"] = std
    
    return results, plot_data, metrics_keys


In [ ]:
results, plot_data , metrics_keys = run_inprocessing(
    df_cleaned, n_folds, repair_levels,
    features, protected_attributes,
    models
)
for model_name in models.keys():
    print_fairness_results_table(plot_data[model_name], metrics_keys, repair_levels)
    for metric in ['accuracy', 'f1', 'roc_auc', 'demographic_parity_ratio', 'equalized_odds_ratio']:
        plot_metrics(plot_data[model_name], metric, repair_levels, protected_attributes)
plot_metrics_grouped(results, protected_attributes=protected_attributes, repair_levels=repair_levels)